In [2]:
import sqlite3
import pandas as pd
import os
from typing import Optional
from datetime import datetime

In [21]:
DB_PATH = "ledger.db"
GENERATOR_DB_PATH = "Generator_Dataset.xlsx"
GENERATOR_DB = "Generator_Dataset.csv"
VENDOR_DB_PATH = "Vendor_Dataset.xlsx"
VENDOR_DB = "Vendor_Dataset.csv"
BOOKINGS_PATH = "bookings.csv"
BOOKING_ITEMS_PATH = "booking_items.csv"

In [4]:
def parse_day_month_to_full(s, default_time="08:00"):
    s = s.strip()
    if s == "" or s is None:
        return None
    s = s.replace("/", "-")
    parts = s.split()
    date_part = parts[0]
    time_part = parts[1] if len(parts) > 1 else None
    now = datetime.now()
    date_chunks = date_part.split("-")
    if len(date_chunks) == 2:
        day, month = date_chunks
        year = now.year
    elif len(date_chunks) == 3:
        a,b,c = date_chunks
        if len(a) == 4:
            year, month, day = a,b,c
        else:
            day, month, year = a,b,c
    else:
        raise ValueError("Unsupported date format: " + s)
    if time_part is None:
        time_part = default_time
    dt_str = f"{int(year):04d}-{int(month):02d}-{int(day):02d} {time_part}"
    datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
    return dt_str

In [5]:
def parse_dt_or_daymonth(s, default_time="08:00"):
    s = (s or "").strip()
    if s == "":
        return None
    try:
        dt = datetime.strptime(s, "%Y-%m-%d %H:%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return parse_day_month_to_full(s, default_time=default_time)

In [6]:
def parse_dt_or_daymonth(s: Optional[str], default_time: str = "08:00"):
    """Try to parse full datetime first; if fails, try day-month formats."""
    s = (s or "").strip()
    if s == "":
        return None
    # try full parse
    try:
        dt = datetime.strptime(s, "%Y-%m-%d %H:%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except Exception:
        # try day-month
        return parse_day_month(s, default_time=default_time)

In [17]:
# Create tables in Dataset
def init_db(conn):
    cur = conn.cursor()
    cur.executescript("""
    CREATE TABLE IF NOT EXISTS generators (
        generator_id TEXT PRIMARY KEY,
        capacity_kva INTEGER,
        identification TEXT,
        type TEXT,
        status TEXT,
        notes TEXT
    );
    CREATE TABLE IF NOT EXISTS vendors (
        vendor_id TEXT PRIMARY KEY,
        vendor_name TEXT,
        vendor_place TEXT,
        phone TEXT
    );
    CREATE TABLE IF NOT EXISTS bookings (
        booking_id TEXT PRIMARY KEY,
        vendor_id TEXT,
        created_at TEXT,
        status TEXT,
        FOREIGN KEY(vendor_id) REFERENCES vendors(vendor_id)
    );
    CREATE TABLE IF NOT EXISTS booking_items (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        booking_id TEXT,
        generator_id TEXT,
        start_dt TEXT,
        end_dt TEXT,
        item_status TEXT,
        remarks TEXT,
        FOREIGN KEY(booking_id) REFERENCES bookings(booking_id),
        FOREIGN KEY(generator_id) REFERENCES generators(generator_id)
    );
    """)
    conn.commit()

In [19]:
def load_sample_data(conn):
    cur = conn.cursor()

    if os.path.exists(GENERATOR_DB_PATH):
        gdf = pd.read_excel(GENERATOR_DB_PATH)
        gdf.to_csv(GENERATOR_DB, index=False)

    if os.path.exists(VENDOR_DB_PATH):
        vdf = pd.read_excel(VENDOR_DB_PATH)
        vdf.to_csv(VENDOR_DB, index=False)

    for _, r in gdf.iterrows():
        cur.execute("""INSERT OR REPLACE INTO generators(generator_id, capacity_kva, identification, type, status, notes)
                       VALUES (?, ?, ?, ?, ?, ?)""",
                    (r["Generator_ID"], int(r["Capacity_KVA"]), r.get("Identification ",""), r.get("Type",""), r.get("Status","Active"), r.get("Notes","")))
    for _, r in vdf.iterrows():
        cur.execute("""INSERT OR REPLACE INTO vendors(vendor_id, vendor_name, vendor_place, phone)
                       VALUES (?, ?, ?, ?)""",
                    (r["Vendor_ID"], r.get("Vendor_Name",""), r.get("Vendor_Place",""), r.get("Phone","")))
    conn.commit()

In [8]:
def is_generator_available(conn, generator_id, new_start, new_end):
    ns = datetime.strptime(new_start, "%Y-%m-%d %H:%M")
    ne = datetime.strptime(new_end, "%Y-%m-%d %H:%M")
    cur = conn.cursor()
    q = """
    SELECT bi.id, bi.booking_id, bi.start_dt, bi.end_dt
    FROM booking_items bi
    JOIN bookings b ON bi.booking_id = b.booking_id
    WHERE bi.generator_id = ? AND bi.item_status = 'Confirmed' AND b.status = 'Confirmed'
    """
    cur.execute(q, (generator_id,))
    rows = cur.fetchall()
    for row in rows:
        bs = datetime.strptime(row[2], "%Y-%m-%d %H:%M")
        be = datetime.strptime(row[3], "%Y-%m-%d %H:%M")
        if ns < be and ne > bs:
            return False, {"conflict_with": row[1], "existing_start": row[2], "existing_end": row[3]}
    return True, None

In [9]:
def find_available_generators(conn, capacity_kva, start_dt, end_dt, needed=1):
    cur = conn.cursor()
    cur.execute("SELECT generator_id FROM generators WHERE capacity_kva = ? AND status = 'Active'", (capacity_kva,))
    candidates = [r[0] for r in cur.fetchall()]
    free_list = []
    for gid in candidates:
        avail, info = is_generator_available(conn, gid, start_dt, end_dt)
        if avail:
            free_list.append(gid)
            if len(free_list) >= needed:
                break
    return free_list

In [10]:
def create_booking(conn, booking_id, vendor_id, items):
    """
    items: list of dicts -> {"generator_id": "...", "start_dt": "...", "end_dt": "...", "remarks": "..."} OR
           allow generator_id to be None and accept 'capacity_kva' to auto-assign
    """
    cur = conn.cursor()
    # create booking row
    cur.execute("INSERT OR REPLACE INTO bookings(booking_id, vendor_id, created_at, status) VALUES (?, ?, ?, ?)",
                (booking_id, vendor_id, datetime.utcnow().isoformat(), "Confirmed"))
    # for each item, verify availability and insert booking_items
    for it in items:
        gen = it.get("generator_id")
        start_dt = it.get("start_dt")
        end_dt = it.get("end_dt")
        remarks = it.get("remarks","")
        if gen is None:
            # attempt auto-assign by capacity if capacity_kva provided
            cap = it.get("capacity_kva")
            if cap is None:
                raise ValueError("Either generator_id or capacity_kva must be provided for each item")
            free = find_available_generators(conn, int(cap), start_dt, end_dt, needed=1)
            if not free:
                raise RuntimeError(f"No available generator of {cap} kVA for {start_dt} - {end_dt}")
            gen = free[0]
        else:
            # check availability for this generator
            available, conflict = is_generator_available(conn, gen, start_dt, end_dt)
            if not available:
                raise RuntimeError(f"Generator {gen} not available for {start_dt} - {end_dt}. Conflict: {conflict}")
        cur.execute("""INSERT INTO booking_items(booking_id, generator_id, start_dt, end_dt, item_status, remarks)
                       VALUES (?, ?, ?, ?, ?, ?)""", (booking_id, gen, start_dt, end_dt, "Confirmed", remarks))
    conn.commit()

In [11]:
def add_generator_to_booking(conn, booking_id, generator_id=None, capacity_kva=None, start_dt=None, end_dt=None, remarks="Added"):
    # add single generator to an existing booking (for last-minute increase)
    # If generator_id is None, auto-assign using capacity_kva
    cur = conn.cursor()
    # fetch booking exists and vendor
    cur.execute("SELECT booking_id, vendor_id FROM bookings WHERE booking_id = ?", (booking_id,))
    r = cur.fetchone()
    if not r:
        raise RuntimeError("Booking not found: " + booking_id)
    if generator_id is None:
        if capacity_kva is None:
            raise ValueError("Provide generator_id or capacity_kva for assignment")
        free = find_available_generators(conn, int(capacity_kva), start_dt, end_dt, needed=1)
        if not free:
            return False, f"No free generator of {capacity_kva} kVA for {start_dt} - {end_dt}"
        generator_id = free[0]
    else:
        available, conflict = is_generator_available(conn, generator_id, start_dt, end_dt)
        if not available:
            return False, f"Generator {generator_id} not available: {conflict}"
    cur.execute("""INSERT INTO booking_items(booking_id, generator_id, start_dt, end_dt, item_status, remarks)
                   VALUES (?, ?, ?, ?, ?, ?)""", (booking_id, generator_id, start_dt, end_dt, "Confirmed", remarks))
    conn.commit()
    return True, generator_id

In [12]:
def modify_booking_times(conn, booking_id, new_start_dt, new_end_dt):
    # Update all booking_items of booking_id to the new time window (common case).
    # For each generator assigned, check conflicts with other bookings.
    cur = conn.cursor()
    cur.execute("SELECT generator_id FROM booking_items WHERE booking_id = ? AND item_status = 'Confirmed'", (booking_id,))
    assigned = [r[0] for r in cur.fetchall()]
    for gid in assigned:
        avail, conflict = is_generator_available(conn, gid, new_start_dt, new_end_dt)
        # Note: is_generator_available will detect the existing booking_item as conflict as well,
        # so we need to ignore the current booking's own items. We'll refine query by excluding rows of this booking.
        if not avail:
            # check if the conflict is only with the same booking (safe) or with other booking
            # We will check manually using SQL to exclude booking_id
            q = """
            SELECT bi.booking_id, bi.start_dt, bi.end_dt FROM booking_items bi
            JOIN bookings b ON bi.booking_id = b.booking_id
            WHERE bi.generator_id = ? AND bi.booking_id != ? AND bi.item_status = 'Confirmed'
            """
            cur.execute(q, (gid, booking_id))
            other = cur.fetchall()
            conflict_with_other = False
            for row in other:
                bs = parse_dt(row[1]); be = parse_dt(row[2])
                ns = parse_dt(new_start_dt); ne = parse_dt(new_end_dt)
                if ns < be and ne > bs:
                    conflict_with_other = True
                    conflict_row = {"conflict_with": row[0], "existing_start": row[1], "existing_end": row[2]}
                    break
            if conflict_with_other:
                return False, f"Cannot update booking: generator {gid} conflicts with booking {conflict_row}"
    # if all good, update them
    cur.execute("UPDATE booking_items SET start_dt = ?, end_dt = ? WHERE booking_id = ? AND item_status = 'Confirmed'",
                (new_start_dt, new_end_dt, booking_id))
    conn.commit()
    return True, "Updated"

In [13]:
def cancel_booking(conn, booking_id, reason="Cancelled"):
    cur = conn.cursor()
    cur.execute("UPDATE bookings SET status = 'Cancelled' WHERE booking_id = ?", (booking_id,))
    cur.execute("UPDATE booking_items SET item_status = 'Cancelled', remarks = ? WHERE booking_id = ?", (reason, booking_id))
    conn.commit()

In [14]:
def export_csv(conn, out_dir="exported_data"):
    os.makedirs(out_dir, exist_ok=True)
    pd.read_sql_query("SELECT * FROM bookings", conn).to_csv(os.path.join(out_dir,BOOKINGS_PATH), index=False)
    pd.read_sql_query("SELECT * FROM booking_items", conn).to_csv(os.path.join(out_dir,BOOKING_ITEMS_PATH), index=False)
    pd.read_sql_query("SELECT * FROM generators", conn).to_csv(os.path.join(out_dir,GENERATOR_DB), index=False)
    pd.read_sql_query("SELECT * FROM vendors", conn).to_csv(os.path.join(out_dir,VENDOR_DB), index=False)
    return os.path.join(out_dir,BOOKINGS_PATH), os.path.join(out_dir,BOOKING_ITEMS_PATH)

In [15]:
def print_table(conn, table):
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(df.to_string(index=False))

In [20]:
conn = sqlite3.connect(DB_PATH, detect_types=sqlite3.PARSE_DECLTYPES | sqlite3.PARSE_COLNAMES)
init_db(conn)
load_sample_data(conn)
print("Initial data loaded. DB path:", DB_PATH)

UnboundLocalError: cannot access local variable 'gdf' where it is not associated with a value

In [ ]:
booking_id = input('Enter booking id (e.g. BKG-0001): ').strip()
# booking_id = "BKG-0001"
vendor_id = input('Vendor id (e.g. V001): ').strip()
# vendor_id = "V001"
n = int(input('How many generators in this booking? ').strip())
# n = 2
items = []
for i in range(n):
    print(f"Item {i+1}:")
    mode = input("Assign by (1) specific generator_id or (2) capacity_kva? Enter 1 or 2: ").strip()
    if mode == '1':
        gid = input('Generator ID (e.g. GEN-45-01): ').strip()
        start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
        end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
        items.append({'generator_id': gid, 'start_dt': start_dt, 'end_dt': end_dt, 'remarks': ''}) # type: ignore
    else:
        cap = int(input('Capacity kVA (e.g. 45): ').strip())
        start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
        end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
        items.append({'capacity_kva': cap, 'start_dt': start_dt, 'end_dt': end_dt, 'remarks': ''}) # type: ignore
items
# create_booking(conn, booking_id, vendor_id, items)
# print(f"Created booking {booking_id}.")

In [ ]:
items = [{'capacity_kva': 20, 'start_dt': '2025-11-26 08:00', 'end_dt': '2025-11-26 17:00', 'remarks': ''},
 {'capacity_kva': 45, 'start_dt': '2025-11-27 08:00', 'end_dt': '2025-11-27 17:00', 'remarks': ''}]
items

In [ ]:
booking_id = "BKG-0001"
vendor_id = "VEN-001"
create_booking(conn, booking_id, vendor_id, items)
print(f"Created booking {booking_id}.")



In [ ]:
print_table(conn, "booking_items")

In [ ]:
# gdf = pd.read_sql_query("SELECT * FROM booking_items", conn)
gdf = pd.read_csv("Vendor_Dataset.csv")
gdf['Vendor_ID'].describe()

In [ ]:
export_csv(conn, out_dir="exported_data")

In [ ]:
 
# query = '''DELETE FROM booking_items;'''
query = '''DELETE FROM bookings;'''
conn.execute(query).fetchall()

In [ ]:
def main():
    while True:
        print("\nGenerator Booking Ledger - CLI")
        print("1. List generators")
        print("2. List vendors")
        print("3. List bookings and items")
        print("4. Create booking")
        print("5. Add generator to booking (last-minute increase)")
        print("6. Modify booking times")
        print("7. Cancel booking")
        print("8. Export CSVs")
        print("9. Exit")
        choice = input("Choose an option (1-9): ").strip()
        try:
            if choice == '1':
                print_table(conn, 'generators')
            elif choice == '2':
                print_table(conn, 'vendors')
            elif choice == '3':
                print_table(conn, 'bookings')
                print_table(conn, 'booking_items')
            elif choice == '4':
                booking_id = input('Enter booking id (e.g. BKG-0001): ').strip()
                vendor_id = input('Vendor id (e.g. V001): ').strip()
                n = int(input('How many generators in this booking? ').strip())
                items = []
                for i in range(n):
                    print(f"Item {i+1}:")
                    mode = input("Assign by (1) specific generator_id or (2) capacity_kva? Enter 1 or 2: ").strip()
                    if mode == '1':
                        gid = input('Generator ID (e.g. GEN-45-01): ').strip()
                        start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
                        end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
                        items.append({'generator_id': gid, 'start_dt': start_dt, 'end_dt': end_dt, 'remarks': ''}) # type: ignore
                    else:
                        cap = int(input('Capacity kVA (e.g. 45): ').strip())
                        start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
                        end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
                        items.append({'capacity_kva': cap, 'start_dt': start_dt, 'end_dt': end_dt, 'remarks': ''}) # type: ignore
                create_booking(conn, booking_id, vendor_id, items)
                print(f"Created booking {booking_id}.")
            elif choice == '5':
                booking_id = input('Booking id to add generator to: ').strip()
                sub = input('Add by (1) specific generator_id or (2) capacity_kva? Enter 1 or 2: ').strip()
                if sub == '1':
                    gid = input('Generator ID: ').strip()
                    start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
                    end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
                    success, info = add_generator_to_booking(conn, booking_id, generator_id=gid, start_dt=start_dt, end_dt=end_dt) # type: ignore
                else:
                    cap = int(input('Capacity kVA: ').strip())
                    start_dt = input("Start (YYYY-MM-DD HH:MM): ").strip()
                    end_dt = input("End   (YYYY-MM-DD HH:MM): ").strip()
                    success, info = add_generator_to_booking(conn, booking_id, capacity_kva=cap, start_dt=start_dt, end_dt=end_dt) # type: ignore
                if success:
                    print(f'Added generator {info} to booking {booking_id}.')
                else:
                    print('Failed to add generator:', info) # type: ignore
            elif choice == '6':
                booking_id = input('Booking id to modify: ').strip()
                new_start = input("New Start (YYYY-MM-DD HH:MM): ").strip()
                new_end = input("New End   (YYYY-MM-DD HH:MM): ").strip()
                success, msg = modify_booking_times(conn, booking_id, new_start, new_end)
                if success:
                    print('Booking times updated.')
                else:
                    print('Failed to update:', msg)
            elif choice == '7':
                booking_id = input('Booking id to cancel: ').strip()
                cancel_booking(conn, booking_id, reason='Cancelled via CLI')
                print('Booking cancelled.')
            elif choice == '8':
                bpath, ipath = export_csv(conn)
                print('Exported:', bpath, ipath)
            elif choice == '9':
                print('Goodbye.')
                break
            else:
                print('Invalid choice.')
        except Exception as e:
            print('Error:', e)



In [ ]:
if __name__ == '__main__':
    main()

In [ ]:
# Export bookings.csv
df_bookings = pd.read_sql_query("SELECT * FROM bookings", conn)
bookings_path = BOOKINGS_PATH
df_bookings.to_csv(bookings_path, index=False)

In [ ]:
# Export booking_items.csv
df_items = pd.read_sql_query("SELECT * FROM booking_items", conn)
booking_items_path = BOOKING_ITEMS_PATH
df_items.to_csv(booking_items_path, index=False)

In [ ]:
b1 = "BKG-0001"
try:
    create_booking(conn, b1, "VEN004", [
        {"capacity_kva":20, "start_dt":"2025-11-24 08:00", "end_dt":"2025-11-24 23:00", "remarks":"Marriage function"}
    ])
    print("\nCreated booking", b1, "for Vendor VEN004 (one 20kVA auto-assigned).")
except Exception as e:
    print("Failed booking create:", e)

show_tables(conn)

In [31]:

"""Load sample data from Excel files."""
# Load generators
print(f"\n[DEBUG] Attempting to load generators from: {GENERATOR_DB_PATH}")
if os.path.exists(GENERATOR_DB_PATH):
    try:
        print(f"[DEBUG] File exists. Reading Excel file...")
        gdf = pd.read_excel(GENERATOR_DB_PATH)
        print(f"[DEBUG] Successfully read Excel. Columns: {list(gdf.columns)}")
        print(f"[DEBUG] Shape: {gdf.shape}")
        print(f"[DEBUG] First row preview:\n{gdf.head(1)}")
        
        gdf.to_csv(GENERATOR_DB, index=False)
        print(f"[DEBUG] Saved to CSV: {GENERATOR_DB}")
        
        loaded_count = 0
        for idx, row in gdf.iterrows():
            try:
                print(f"[DEBUG] Processing generator row {idx}: {row['Generator_ID']}")
                generator = Generator(
                    generator_id=str(row["Generator_ID"]),
                    capacity_kva=int(row["Capacity_KVA"]),
                    identification=str(row.get("Identification", "")) or "",
                    type=str(row.get("Type", "")) or "",
                    status=str(row.get("Status", GeneratorStatus.ACTIVE.value)) or GeneratorStatus.ACTIVE.value,
                    notes=str(row.get("Notes", "")) or ""
                )
                self.generator_repo.save(generator)
                loaded_count += 1
            except Exception as row_error:
                print(f"[ERROR] Failed to process row {idx}: {row_error}")
        
        print(f"✓ Loaded {loaded_count}/{len(gdf)} generators")
    except Exception as e:
        print(f"[ERROR] Error loading generators: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"[WARNING] Generator dataset not found at {GENERATOR_DB_PATH}")
    print(f"[DEBUG] Current working directory: {os.getcwd()}")
    print(f"[DEBUG] Directory contents: {os.listdir('.')}")

# Load vendors
print(f"\n[DEBUG] Attempting to load vendors from: {VENDOR_DB_PATH}")
if os.path.exists(VENDOR_DB_PATH):
    try:
        print(f"[DEBUG] File exists. Reading Excel file...")
        vdf = pd.read_excel(VENDOR_DB_PATH)
        print(f"[DEBUG] Successfully read Excel. Columns: {list(vdf.columns)}")
        print(f"[DEBUG] Shape: {vdf.shape}")
        print(f"[DEBUG] First row preview:\n{vdf.head(1)}")
        
        vdf.to_csv(VENDOR_DB, index=False)
        print(f"[DEBUG] Saved to CSV: {VENDOR_DB}")
        
        loaded_count = 0
        for idx, row in vdf.iterrows():
            try:
                print(f"[DEBUG] Processing vendor row {idx}: {row['Vendor_ID']}")
                vendor = Vendor(
                    vendor_id=str(row["Vendor_ID"]),
                    vendor_name=str(row.get("Vendor_Name", "")) or "",
                    vendor_place=str(row.get("Vendor_Place", "")) or "",
                    phone=str(row.get("Phone", "")) or ""
                )
                self.vendor_repo.save(vendor)
                loaded_count += 1
            except Exception as row_error:
                print(f"[ERROR] Failed to process row {idx}: {row_error}")
        
        print(f"✓ Loaded {loaded_count}/{len(vdf)} vendors")
    except Exception as e:
        print(f"[ERROR] Error loading vendors: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"[WARNING] Vendor dataset not found at {VENDOR_DB_PATH}")
    print(f"[DEBUG] Current working directory: {os.getcwd()}")
    print(f"[DEBUG] Directory contents: {os.listdir('.')}")


[DEBUG] Attempting to load generators from: Data/Generator_Dataset.xlsx
[DEBUG] File exists. Reading Excel file...
[DEBUG] Successfully read Excel. Columns: ['Generator_ID', 'Capacity_KVA', 'Identification ', 'Type', 'Status', 'Notes']
[DEBUG] Shape: (29, 6)
[DEBUG] First row preview:
         Generator_ID  Capacity_KVA Identification  Type  Status Notes
0  GEN-20KVA-PL-HA-01            20     Plain(सादा)   HA  Active     -
[DEBUG] Saved to CSV: Generator_Dataset.csv
[DEBUG] Processing generator row 0: GEN-20KVA-PL-HA-01
[ERROR] Failed to process row 0: name 'Generator' is not defined
[DEBUG] Processing generator row 1: GEN-20KVA-PL-HARA-02
[ERROR] Failed to process row 1: name 'Generator' is not defined
[DEBUG] Processing generator row 2: GEN-20KVA-RD-RBTA-03
[ERROR] Failed to process row 2: name 'Generator' is not defined
[DEBUG] Processing generator row 3: GEN-20KVA-BU-RB-04
[ERROR] Failed to process row 3: name 'Generator' is not defined
[DEBUG] Processing generator row 4: GEN-20K

In [30]:
load_from_excel

<function __main__.load_from_excel(self) -> None>